# k-Nearest Neighbors Classification On Handwritten Digits

**Purpose:** classify handwritten digit images with k-nearest neighbors while keeping scaling inside validation folds.

**Dataset:** `sklearn.datasets.load_digits()` with 8x8 grayscale digit images flattened into 64 numeric features.

**Method:** stratified train/test split, fold-safe `Pipeline(StandardScaler(), KNeighborsClassifier())`, cross-validated `k` selection, final holdout evaluation, and a clearly labeled PCA visualization.

**Metric:** accuracy and macro F1; digits are close to balanced, but macro F1 keeps each digit visible.

**Headline takeaway:** `k=5` reaches holdout macro F1 `0.964` on this benchmark, and the PCA visualization is interpretive rather than the evaluated model.


## Imports And Data Load

Each row is an 8-by-8 image flattened into 64 pixel-intensity features.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42
digits = load_digits()
X = digits.data
y = digits.target

print(f"Rows: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(digits.target_names)}")


## Inspect Class Balance And Example Images

The target is nearly balanced, so accuracy is a reasonable headline metric. The sample grid keeps the image task concrete for readers.


In [ ]:
class_counts = pd.Series(y, name="digit").value_counts().sort_index().rename("count")
display(class_counts.to_frame())

fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for digit, ax in enumerate(axes.ravel()):
    sample_index = np.flatnonzero(y == digit)[0]
    ax.imshow(digits.images[sample_index], cmap="gray_r")
    ax.set_title(f"Digit {digit}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Create A Stratified Holdout Split

The final test split is untouched during `k` selection.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]} | Test rows: {X_test.shape[0]}")


## Define The kNN Pipeline

Scaling is inside the pipeline so every cross-validation fold learns scale parameters only from its own training fold.


In [ ]:
def make_knn_pipeline(k):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k)),
        ]
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
k_values = [1, 3, 5, 7, 9, 11, 15]


## Tune The Number Of Neighbors

The validation curve shows the accuracy/stability tradeoff as `k` grows.


In [ ]:
rows = []
for k in k_values:
    scores = cross_val_score(make_knn_pipeline(k), X_train, y_train, cv=cv, scoring="accuracy")
    rows.append({"k": k, "cv_accuracy_mean": scores.mean(), "cv_accuracy_std": scores.std()})

cv_results = pd.DataFrame(rows).sort_values(
    ["cv_accuracy_mean", "cv_accuracy_std", "k"],
    ascending=[False, True, True],
)
display(cv_results)

best_k = int(cv_results.iloc[0]["k"])
print(f"Selected k: {best_k}")


## Plot The Validation Curve

The plot is separated from selection so the numerical ranking remains easy to audit.


In [ ]:
fig, ax = plt.subplots()
k_values = [row["k"] for row in rows]
means = [row["cv_accuracy_mean"] for row in rows]
stds = [row["cv_accuracy_std"] for row in rows]
ax.errorbar(k_values, means, yerr=stds, marker="o", capsize=3, color=ACCENT, linewidth=2)
best_row = max(rows, key=lambda row: row["cv_accuracy_mean"])
ax.scatter([best_row["k"]], [best_row["cv_accuracy_mean"]], color=HIGHLIGHT, s=70, zorder=3)
ax.set_xlabel("k neighbors")
ax.set_ylabel("Mean CV accuracy")
ax.set_title("Small-neighborhood kNN performs best on handwritten digits")
ax.text(
    0,
    -0.22,
    "Digits training split; scaling is inside each cross-validation fold.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "digits_knn_validation_curve", project_root=PROJECT_ROOT)
plt.show()

## Final Holdout Evaluation

The selected `k` is refit on the full training split and evaluated once on the test split.


In [ ]:
final_model = make_knn_pipeline(best_k)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Test accuracy: {accuracy:.3f}")
print(f"Test macro F1: {macro_f1:.3f}")


## Confusion Matrix

The heatmap identifies which digit pairs are most frequently confused.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("kNN confusion matrix on held-out digits")
plt.tight_layout()
plt.show()


## Error Review

A small gallery of mistakes is more useful than a metric alone because it shows the visual ambiguity in the data.


In [ ]:
error_indices = np.flatnonzero(y_test != y_pred)
print(f"Misclassified test images: {len(error_indices)}")

if len(error_indices):
    shown = error_indices[:10]
    fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
    for plot_index, ax in zip(shown, axes.ravel()):
        ax.imshow(X_test[plot_index].reshape(8, 8), cmap="gray_r")
        ax.set_title(f"true {y_test[plot_index]} / pred {y_pred[plot_index]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples in this split.")


## PCA Visualization

This two-dimensional model is only for visualization. The evaluated model above uses all 64 pixel features.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

pca_knn = KNeighborsClassifier(n_neighbors=best_k)
pca_knn.fit(X_train_pca, y_train)
print(f"PCA explained variance in 2D view: {pca.explained_variance_ratio_.sum():.3f}")


## Plot PCA Decision Regions

The regions make local-neighborhood behavior visible, but they should not be read as the final classifier's performance surface.


In [ ]:
x_min, x_max = X_test_pca[:, 0].min() - 1, X_test_pca[:, 0].max() + 1
y_min, y_max = X_test_pca[:, 1].min() - 1, X_test_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
Z = pca_knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.20, levels=np.arange(-0.5, 10.5, 1), cmap="tab10")
scatter = plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap="tab10", edgecolor="k", s=35)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Digits test samples in PCA space with kNN regions")
plt.colorbar(scatter, ticks=range(10), label="Digit")
plt.tight_layout()
plt.show()


## Conclusion

The notebook keeps preprocessing inside validation folds, reports both accuracy and macro F1, and separates the evaluated 64-feature classifier from the explanatory PCA view. The limitation is that kNN is memory-based and this benchmark is small compared with modern image-classification settings.
